In [ ]:
#  PG-HWLD: 17,160 samples, 26 balanced classes (A-Z, case-merged)
#  660 samples/class, 28×28 grayscale PNG, white-on-black.
#  Smallest dataset in this family — highest dropout, strongest augmentation,
#  longest patience to avoid stopping too early.
#
#  TWO MODES:
#    MODE = "standalone"     — split PG-HWLD into train/val/test internally (this is implemented in this)
#    MODE = "external_test"  — train on EMNIST/letters, test on PG-HWLD
#
#  Set DATASET_ROOT to the folder containing the "resources" sub-tree, OR
#  set IMAGE_DIR directly to the folder containing per-letter subfolders.

In [ ]:
# importing necessary dependenies
import os, random, json, glob
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

MODE         = "standalone"          # "standalone" | "external_test"
DATASET_ROOT = "/kaggle/input/datasets/rautranjit/pghwld/PG-HWLD/"
IMAGE_DIR    = os.path.join(DATASET_ROOT,
               "resources", "datasets", "dataset-processed", "test-images")

NUM_CLASSES  = 26
IMG          = 28
BS           = 32            # small batch: only 660 samples/class
EPOCHS       = 100
LR           = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1          # independent handwriters → more style variance
WARMUP_EP    = 6
VAL_SPLIT    = 0.12
TEST_SPLIT   = 0.10          # standalone mode only
RESULTS_DIR  = "./results/pg_hwld"
AUTOTUNE     = tf.data.AUTOTUNE

LABELS = [chr(c) for c in range(65, 91)]   # ['A', 'B', ..., 'Z']
os.makedirs(RESULTS_DIR, exist_ok=True)

#  DATA LOADING  —  read PNGs from per-letter subdirectories
def load_pg_hwld(image_dir: str):
    """
    Walk <image_dir>/<letter>/*.png and return (paths, labels) arrays.
    Labels are 0-indexed ints (0=A … 25=Z).
    Accepts both lowercase ('a') and uppercase ('A') folder names.
    """
    paths, labels = [], []
    for entry in sorted(os.listdir(image_dir)):
        letter = entry.strip().upper()
        if len(letter) != 1 or not letter.isalpha():
            continue
        label  = ord(letter) - ord('A')
        folder = os.path.join(image_dir, entry)
        pngs   = sorted(glob.glob(os.path.join(folder, "*.png")))
        if not pngs:
            pngs = sorted(glob.glob(os.path.join(folder, "*.*")))
        paths.extend(pngs)
        labels.extend([label] * len(pngs))
        print(f"  [DATA]  '{letter}' (label {label:>2}) : {len(pngs):>5} samples")
    return np.array(paths), np.array(labels, dtype=np.int32)

print(f"[INFO] Scanning {IMAGE_DIR} …")
all_paths, all_labels = load_pg_hwld(IMAGE_DIR)
print(f"[INFO] Total samples : {len(all_paths):,} across {NUM_CLASSES} classes")

idx = np.random.permutation(len(all_paths))
all_paths, all_labels = all_paths[idx], all_labels[idx]

if MODE == "standalone":
    n_test  = int(len(all_paths) * TEST_SPLIT)
    n_val   = int(len(all_paths) * VAL_SPLIT)
    n_train = len(all_paths) - n_test - n_val
    test_paths,  test_labels  = all_paths[:n_test],              all_labels[:n_test]
    val_paths,   val_labels   = all_paths[n_test:n_test+n_val],  all_labels[n_test:n_test+n_val]
    train_paths, train_labels = all_paths[n_test+n_val:],        all_labels[n_test+n_val:]
    print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {n_test:,}")

elif MODE == "external_test":
    try:
        import tensorflow_datasets as tfds
    except ImportError:
        raise ImportError("MODE='external_test' requires tensorflow_datasets.")
    print("[INFO] Loading EMNIST/letters for training (external_test mode) …")
    emnist_train_raw, info = tfds.load(
        "emnist/letters", split="train", as_supervised=True,
        with_info=True, shuffle_files=True,
    )
    total_em   = info.splits["train"].num_examples
    n_val_em   = int(total_em * VAL_SPLIT)
    n_train_em = total_em - n_val_em
    emnist_train_raw = emnist_train_raw.shuffle(10_000, seed=SEED,
                                                reshuffle_each_iteration=False)
    val_raw_em   = emnist_train_raw.take(n_val_em)
    train_raw_em = emnist_train_raw.skip(n_val_em)
    print(f"[INFO] EMNIST Train: {n_train_em:,} | Val: {n_val_em:,} | "
          f"PG-HWLD test: {len(all_paths):,}")
    test_paths, test_labels = all_paths, all_labels
else:
    raise ValueError(f"Unknown MODE: {MODE!r}")

#  PREPROCESSING  —  decode PNG -> float32 in [-1, 1]
def load_and_preprocess_png(path, label):
    raw   = tf.io.read_file(path)
    image = tf.image.decode_png(raw, channels=1)
    image = tf.image.resize(image, [IMG, IMG])
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    return image, label

def augment(image, label):
    # Stronger augmentation: fewer samples -> more regularisation needed
    image = tf.image.random_brightness(image, 0.30)
    image = tf.image.random_contrast(image, 0.70, 1.30)
    image = tf.pad(image, [[4, 4], [4, 4], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    # Elastic-style local displacement
    dy = tf.random.uniform([], -2, 2, dtype=tf.int32)
    dx = tf.random.uniform([], -2, 2, dtype=tf.int32)
    image = tf.pad(image, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    image = image[2+dy:2+dy+IMG, 2+dx:2+dx+IMG, :]
    image = tf.ensure_shape(image, [IMG, IMG, 1])
    image = image + tf.random.normal(tf.shape(image), stddev=0.04)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

if MODE == "standalone":
    train_path_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
    val_path_ds   = tf.data.Dataset.from_tensor_slices((val_paths,   val_labels))
    test_path_ds  = tf.data.Dataset.from_tensor_slices((test_paths,  test_labels))

    train_ds = (
        train_path_ds
        .shuffle(n_train, seed=SEED, reshuffle_each_iteration=True)
        .map(load_and_preprocess_png, num_parallel_calls=AUTOTUNE)
        .map(augment,                 num_parallel_calls=AUTOTUNE)
        .map(to_onehot,               num_parallel_calls=AUTOTUNE)
        .batch(BS).prefetch(AUTOTUNE)
    )
    val_ds = (
        val_path_ds
        .map(load_and_preprocess_png, num_parallel_calls=AUTOTUNE)
        .map(to_onehot,               num_parallel_calls=AUTOTUNE)
        .batch(BS).prefetch(AUTOTUNE)
    )
    test_ds = (
        test_path_ds
        .map(load_and_preprocess_png, num_parallel_calls=AUTOTUNE)
        .batch(BS).prefetch(AUTOTUNE)
    )
    test_ds_oh = (
        test_path_ds
        .map(load_and_preprocess_png, num_parallel_calls=AUTOTUNE)
        .map(to_onehot,               num_parallel_calls=AUTOTUNE)
        .batch(BS).prefetch(AUTOTUNE)
    )
    steps_per_epoch = max(1, n_train // BS)

else:  # external_test
    def preprocess_emnist(image, label):
        image = tf.cast(image, tf.float32) / 127.5 - 1.0
        label = tf.cast(label, tf.int32) - 1   # 1-26 → 0-25
        return image, label

    train_ds = (
        train_raw_em
        .map(preprocess_emnist, num_parallel_calls=AUTOTUNE)
        .map(augment,           num_parallel_calls=AUTOTUNE)
        .map(to_onehot,         num_parallel_calls=AUTOTUNE)
        .batch(BS).prefetch(AUTOTUNE)
    )
    val_ds = (
        val_raw_em
        .map(preprocess_emnist, num_parallel_calls=AUTOTUNE)
        .map(to_onehot,         num_parallel_calls=AUTOTUNE)
        .batch(BS).prefetch(AUTOTUNE)
    )
    test_path_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
    test_ds = (
        test_path_ds
        .map(load_and_preprocess_png, num_parallel_calls=AUTOTUNE)
        .batch(BS).prefetch(AUTOTUNE)
    )
    test_ds_oh = (
        test_path_ds
        .map(load_and_preprocess_png, num_parallel_calls=AUTOTUNE)
        .map(to_onehot,               num_parallel_calls=AUTOTUNE)
        .batch(BS).prefetch(AUTOTUNE)
    )
    steps_per_epoch = max(1, n_train_em // BS)

total_steps = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

#  BUILDING BLOCKS
def gelu(x): return tf.nn.gelu(x)

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, r]); x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch)
    r2  = residual_block(r1, out_ch)
    r3  = residual_block(r2, out_ch)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    return out

def adaptive_filter_capsule(x, num_classes, capsule_dim=32):
    """
    Lightweight capsule routing.
    capsule_dim=32 — small dataset with high style variance; richer routing
    compensates for the limited number of training samples per class.
    """
    h = layers.Dense(256, activation=gelu)(x)
    h = layers.Dense(num_classes * capsule_dim)(h)
    h = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps = layers.Multiply()([x_sliced, h])
    caps = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    return layers.BatchNormalization()(caps)

#  MODEL  —  higher dropout (0.40) for small-dataset regime
def build_model(num_classes=26, image_size=28):
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # Stem
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem); stem = layers.Activation(gelu)(stem)

    #  Encoder
    enc1 = dense_res_block(stem, 64,  64)
    enc2 = dense_res_block(enc1, 64,  128)
    enc3 = dense_res_block(enc2, 128, 256)

    #  Multi-scale GAP fusion
    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.GlobalAveragePooling2D()(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])  # (B, 512)

    #  Adaptive Filter Capsule
    afc_out = adaptive_filter_capsule(fused_gap, K, capsule_dim=32)   # (B, K)

    #  Dense head  (Dropout=0.40: +0.05 vs EMNIST for small-data regime) ─
    x = layers.Dense(128, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation(gelu, name="head_act")(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(K, name="head_logits")(x)

    # Gated fusion
    combined   = layers.Concatenate(name="gate_input")([x, afc_out])
    gate       = layers.Dense(2, activation="softmax", name="gate")(combined)
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x, gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc")([afc_out, gate])
    out = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=out, name="WhatNet_PG_HWLD")

#  LR SCHEDULE
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base         = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

#  TRAIN
model        = build_model(NUM_CLASSES, IMG)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=20,   # long patience: small dataset
                      restore_best_weights=True, verbose=1),
    ],
)

#  EVALUATE
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "PG_HWLD", "mode": MODE, "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)
print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

In [ ]:
2026-04-26 00:51:59.103553: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
E0000 00:00:1777164719.289212      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777164719.341724      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777164719.803521      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777164719.803563      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777164719.803566      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777164719.803568      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
[INFO] Scanning /kaggle/input/datasets/rautranjit/pghwld/PG-HWLD/resources/datasets/dataset-processed/test-images …
  [DATA]  'A' (label  0) :   660 samples
  [DATA]  'B' (label  1) :   660 samples
  [DATA]  'C' (label  2) :   660 samples
  [DATA]  'D' (label  3) :   660 samples
  [DATA]  'E' (label  4) :   660 samples
  [DATA]  'F' (label  5) :   660 samples
  [DATA]  'G' (label  6) :   660 samples
  [DATA]  'H' (label  7) :   660 samples
  [DATA]  'I' (label  8) :   660 samples
  [DATA]  'J' (label  9) :   660 samples
  [DATA]  'K' (label 10) :   660 samples
  [DATA]  'L' (label 11) :   660 samples
  [DATA]  'M' (label 12) :   660 samples
  [DATA]  'N' (label 13) :   660 samples
  [DATA]  'O' (label 14) :   660 samples
  [DATA]  'P' (label 15) :   660 samples
  [DATA]  'Q' (label 16) :   660 samples
  [DATA]  'R' (label 17) :   660 samples
  [DATA]  'S' (label 18) :   660 samples
  [DATA]  'T' (label 19) :   660 samples
  [DATA]  'U' (label 20) :   660 samples
  [DATA]  'V' (label 21) :   660 samples
  [DATA]  'W' (label 22) :   660 samples
  [DATA]  'X' (label 23) :   660 samples
  [DATA]  'Y' (label 24) :   660 samples
  [DATA]  'Z' (label 25) :   660 samples
[INFO] Total samples : 17,160 across 26 classes
[INFO] Train: 13,385 | Val: 2,059 | Test: 1,716
I0000 00:00:1777164743.644434      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777164743.650780      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
[INFO] Steps/epoch: 418
[INFO] Params: 5,496,316
Epoch 1/100
WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1777164767.446791     126 service.cc:152] XLA service 0x7edba8016ff0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777164767.446824     126 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777164767.446828     126 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777164771.421124     126 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777164791.497066     126 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
419/419 ━━━━━━━━━━━━━━━━━━━━ 96s 121ms/step - accuracy: 0.0534 - loss: 3.4429 - val_accuracy: 0.0359 - val_loss: 3.2736
Epoch 2/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 44ms/step - accuracy: 0.2829 - loss: 2.6274 - val_accuracy: 0.6153 - val_loss: 1.7339
Epoch 3/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.6341 - loss: 1.8114 - val_accuracy: 0.8485 - val_loss: 1.1436
Epoch 4/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.7857 - loss: 1.3694 - val_accuracy: 0.8805 - val_loss: 1.0284
Epoch 5/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 0.8341 - loss: 1.2039 - val_accuracy: 0.8951 - val_loss: 1.0339
Epoch 6/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.8497 - loss: 1.1372 - val_accuracy: 0.9038 - val_loss: 0.9633
Epoch 7/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.8780 - loss: 1.0687 - val_accuracy: 0.8902 - val_loss: 0.9913
Epoch 8/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 46ms/step - accuracy: 0.8898 - loss: 1.0353 - val_accuracy: 0.9228 - val_loss: 0.9170
Epoch 9/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9053 - loss: 0.9787 - val_accuracy: 0.9228 - val_loss: 0.9144
Epoch 10/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.9179 - loss: 0.9384 - val_accuracy: 0.9393 - val_loss: 0.8717
Epoch 11/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9231 - loss: 0.9207 - val_accuracy: 0.9228 - val_loss: 0.9037
Epoch 12/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.9285 - loss: 0.9041 - val_accuracy: 0.9480 - val_loss: 0.8266
Epoch 13/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.9338 - loss: 0.8763 - val_accuracy: 0.9490 - val_loss: 0.8172
Epoch 14/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9454 - loss: 0.8497 - val_accuracy: 0.9242 - val_loss: 0.8947
Epoch 15/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9389 - loss: 0.8494 - val_accuracy: 0.9461 - val_loss: 0.8569
Epoch 16/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9485 - loss: 0.8385 - val_accuracy: 0.9388 - val_loss: 0.8719
Epoch 17/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9506 - loss: 0.8202 - val_accuracy: 0.9432 - val_loss: 0.8352
Epoch 18/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9506 - loss: 0.8047 - val_accuracy: 0.9480 - val_loss: 0.8291
Epoch 19/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.9580 - loss: 0.7929 - val_accuracy: 0.9524 - val_loss: 0.8037
Epoch 20/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.9582 - loss: 0.7907 - val_accuracy: 0.9563 - val_loss: 0.8075
Epoch 21/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9598 - loss: 0.7833 - val_accuracy: 0.9500 - val_loss: 0.7869
Epoch 22/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9634 - loss: 0.7667 - val_accuracy: 0.9451 - val_loss: 0.8247
Epoch 23/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9604 - loss: 0.7738 - val_accuracy: 0.9490 - val_loss: 0.7991
Epoch 24/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9648 - loss: 0.7591 - val_accuracy: 0.9524 - val_loss: 0.7937
Epoch 25/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9679 - loss: 0.7536 - val_accuracy: 0.9563 - val_loss: 0.7977
Epoch 26/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.9683 - loss: 0.7494 - val_accuracy: 0.9573 - val_loss: 0.8000
Epoch 27/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.9704 - loss: 0.7351 - val_accuracy: 0.9602 - val_loss: 0.7727
Epoch 28/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9684 - loss: 0.7419 - val_accuracy: 0.9534 - val_loss: 0.8040
Epoch 29/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9707 - loss: 0.7310 - val_accuracy: 0.9466 - val_loss: 0.8232
Epoch 30/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9769 - loss: 0.7202 - val_accuracy: 0.9534 - val_loss: 0.7725
Epoch 31/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.9736 - loss: 0.7251 - val_accuracy: 0.9636 - val_loss: 0.7611
Epoch 32/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9786 - loss: 0.7130 - val_accuracy: 0.9471 - val_loss: 0.8021
Epoch 33/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9758 - loss: 0.7155 - val_accuracy: 0.9602 - val_loss: 0.7695
Epoch 34/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9820 - loss: 0.7039 - val_accuracy: 0.9626 - val_loss: 0.7513
Epoch 35/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9818 - loss: 0.6971 - val_accuracy: 0.9631 - val_loss: 0.7609
Epoch 36/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.9831 - loss: 0.6940 - val_accuracy: 0.9645 - val_loss: 0.7456
Epoch 37/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9843 - loss: 0.6952 - val_accuracy: 0.9543 - val_loss: 0.7827
Epoch 38/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9774 - loss: 0.7040 - val_accuracy: 0.9631 - val_loss: 0.7470
Epoch 39/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9844 - loss: 0.6912 - val_accuracy: 0.9631 - val_loss: 0.7551
Epoch 40/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 0.9850 - loss: 0.6852 - val_accuracy: 0.9650 - val_loss: 0.7514
Epoch 41/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9857 - loss: 0.6824 - val_accuracy: 0.9558 - val_loss: 0.7601
Epoch 42/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9846 - loss: 0.6865 - val_accuracy: 0.9563 - val_loss: 0.7642
Epoch 43/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 46ms/step - accuracy: 0.9850 - loss: 0.6840 - val_accuracy: 0.9650 - val_loss: 0.7627
Epoch 44/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9912 - loss: 0.6706 - val_accuracy: 0.9616 - val_loss: 0.7498
Epoch 45/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 0.9850 - loss: 0.6896 - val_accuracy: 0.9660 - val_loss: 0.7437
Epoch 46/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9915 - loss: 0.6681 - val_accuracy: 0.9645 - val_loss: 0.7591
Epoch 47/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 0.9879 - loss: 0.6777 - val_accuracy: 0.9665 - val_loss: 0.7463
Epoch 48/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 0.9905 - loss: 0.6646 - val_accuracy: 0.9689 - val_loss: 0.7369
Epoch 49/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 0.9906 - loss: 0.6650 - val_accuracy: 0.9694 - val_loss: 0.7314
Epoch 50/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9933 - loss: 0.6632 - val_accuracy: 0.9645 - val_loss: 0.7439
Epoch 51/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9921 - loss: 0.6634 - val_accuracy: 0.9675 - val_loss: 0.7406
Epoch 52/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9929 - loss: 0.6631 - val_accuracy: 0.9665 - val_loss: 0.7502
Epoch 53/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 0.9911 - loss: 0.6642 - val_accuracy: 0.9713 - val_loss: 0.7330
Epoch 54/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9937 - loss: 0.6585 - val_accuracy: 0.9670 - val_loss: 0.7321
Epoch 55/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9916 - loss: 0.6645 - val_accuracy: 0.9641 - val_loss: 0.7515
Epoch 56/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9925 - loss: 0.6590 - val_accuracy: 0.9655 - val_loss: 0.7410
Epoch 57/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9936 - loss: 0.6605 - val_accuracy: 0.9650 - val_loss: 0.7557
Epoch 58/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9952 - loss: 0.6546 - val_accuracy: 0.9660 - val_loss: 0.7389
Epoch 59/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9934 - loss: 0.6572 - val_accuracy: 0.9641 - val_loss: 0.7394
Epoch 60/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9947 - loss: 0.6539 - val_accuracy: 0.9665 - val_loss: 0.7435
Epoch 61/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9967 - loss: 0.6498 - val_accuracy: 0.9660 - val_loss: 0.7432
Epoch 62/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9945 - loss: 0.6556 - val_accuracy: 0.9675 - val_loss: 0.7308
Epoch 63/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9953 - loss: 0.6506 - val_accuracy: 0.9660 - val_loss: 0.7457
Epoch 64/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9957 - loss: 0.6511 - val_accuracy: 0.9679 - val_loss: 0.7355
Epoch 65/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9958 - loss: 0.6497 - val_accuracy: 0.9665 - val_loss: 0.7398
Epoch 66/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9946 - loss: 0.6511 - val_accuracy: 0.9704 - val_loss: 0.7296
Epoch 67/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9956 - loss: 0.6504 - val_accuracy: 0.9694 - val_loss: 0.7302
Epoch 68/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9975 - loss: 0.6454 - val_accuracy: 0.9704 - val_loss: 0.7317
Epoch 69/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9960 - loss: 0.6468 - val_accuracy: 0.9679 - val_loss: 0.7324
Epoch 70/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9969 - loss: 0.6452 - val_accuracy: 0.9709 - val_loss: 0.7300
Epoch 71/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9969 - loss: 0.6461 - val_accuracy: 0.9699 - val_loss: 0.7303
Epoch 72/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9969 - loss: 0.6458 - val_accuracy: 0.9699 - val_loss: 0.7369
Epoch 73/100
419/419 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9979 - loss: 0.6430 - val_accuracy: 0.9694 - val_loss: 0.7345
Epoch 73: early stopping
Restoring model weights from the end of the best epoch: 53.

[RESULT] Test Acc : 96.85%
[RESULT] Macro F1 : 96.89%
[RESULT] Test Loss: 0.7304
[RESULT] Params   : 5,496,316

[RESULT] Worst-5 classes:
         'I' (cls  8) = 88.1%
         'J' (cls  9) = 91.8%
         'A' (cls  0) = 92.4%
         'R' (cls 17) = 92.4%
         'M' (cls 12) = 94.3%